In [ ]:
# 1. Refined FG Stock lookup focusing on Series
def get_fg_stock_by_series(master_row):
    series_val = norm(master_row["Series"])

    # Search in Inventory using Series (Material column)
    # We use 'Material' because in your file it often contains the Series code
    r = best_match(df_inventory, series_val, "", series_col="Material")
    if r is not None and not pd.isna(r["Unrestricted"]):
        return r["Unrestricted"]

    # Search in Sheet10
    r = best_match(df_sheet10, series_val, "", series_col="Material")
    if r is not None and not pd.isna(r["Unrestricted"]):
        return r["Unrestricted"]

    # Search in Sheet4
    r = best_match(df_fg, series_val, "", series_col="Series")
    if r is not None and not pd.isna(r["FG Quantity"]):
        return r["FG Quantity"]

    return 0

# 2. Main Loop with Capacity Constraint and Series-Only Focus
plan = []
machine_backlog = {} 
MAX_MINUTES = 1200 # 20 Hours

# Process only unique Series to save time, or iterate normally if each row is unique demand
for i, row in df_master.iterrows():
    series = row["Series"]
    part = row["Part"] # Kept for the final output label
    
    # 3. Use the refined Series-only stock check
    fg_stock = get_fg_stock_by_series(row)
    dispatch = row["Dispatch"] if not pd.isna(row["Dispatch"]) else 0
    available_fg = fg_stock - dispatch
    
    monthly_req = row['Monthely Requirement-jan'] if not pd.isna(row['Monthely Requirement-jan']) else 0
    daily_demand = monthly_req / 28
    
    # 4. Quantity Logic
    if available_fg < daily_demand:
        planned_qty = daily_demand * 2
    elif daily_demand > 0 and daily_demand <= 50:
        planned_qty = 5 * daily_demand
    else:
        planned_qty = daily_demand
        
    if planned_qty <= 0: continue

    # 5. Machine Lookup (Using Series)
    # Note: Ensure 'Part No.' in your tool list contains the Series ID
    machine_row = best_match(df_tool, series, "", series_col="Part No.")
    if machine_row is None: continue
    machine = machine_row["Machine No."]
    
    # 6. Time Calculation
    time_row = best_match(df_prod, series, "", series_col="Part No.")
    if time_row is None or pd.isna(time_row.get("Part Per Hour")): continue
    
    time_hrs = planned_qty / time_row["Part Per Hour"]
    time_mins = time_hrs * 60
    
    # 7. Capacity Filter
    current_used = machine_backlog.get(machine, 0)
    if current_used + time_mins <= MAX_MINUTES:
        machine_backlog[machine] = current_used + time_mins
        plan.append({
            "Machine": machine,
            "Series": series,
            "Part": part,
            "Qty": int(round(planned_qty)),
            "Time_Required_Hrs": round(time_hrs, 2)
        })

In [ ]:
def get_fg_stock_by_series(series):
    s = norm(series)

    r = df_inventory[df_inventory["material"].astype(str).apply(norm) == s]
    if not r.empty and not pd.isna(r.iloc[0]["unrestricted"]):
        return r.iloc[0]["unrestricted"]

    r = df_sheet10[df_sheet10["material"].astype(str).apply(norm) == s]
    if not r.empty and not pd.isna(r.iloc[0]["unrestricted"]):
        return r.iloc[0]["unrestricted"]

    r = df_fg[df_fg["series"].astype(str).apply(norm) == s]
    if not r.empty and not pd.isna(r.iloc[0]["fg_quantity"]):
        return r.iloc[0]["fg_quantity"]

    return 0


In [ ]:
def get_machine_by_series(series):
    s = norm(series)
    r = df_tool[df_tool["part_no"].astype(str).apply(norm) == s]
    if not r.empty:
        return r.iloc[0]["machine_no"]
    return None


In [ ]:
def normalize_columns(df):
    df = df.copy()
    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace(".", "", regex=False)
    )
    return df


In [ ]:
def get_time_required_by_series(series, qty):
    s = norm(series)
    r = df_prod[df_prod["part_no"].astype(str).apply(norm) == s]
    if r.empty:
        return None

    pph = r.iloc[0]["part_per_hour"]
    if pd.isna(pph) or pph <= 0:
        return None

    return qty / pph


In [ ]:
plan = []
machine_backlog = {}
MAX_MINUTES = 1200  # 20 hours

for i, row in df_master.iterrows():

    series = row["series"]
    part = row["part"]  # only for labeling, not logic

    monthly_req = row["monthely_requirement-jan"] if not pd.isna(row["monthely_requirement-jan"]) else 0
    min_req = row["minimum_requirement"] if not pd.isna(row["minimum_requirement"]) else 0
    dispatch = row["dispatch"] if not pd.isna(row["dispatch"]) else 0

    fg_stock = get_fg_stock_by_series(series)
    available_fg = fg_stock - dispatch

    net_req = monthly_req + min_req - available_fg
    if net_req <= 0:
        continue

    daily_demand = monthly_req / 28

    if available_fg < daily_demand:
        planned_qty = daily_demand
    elif daily_demand <= 50:
        planned_qty = min(5 * daily_demand, net_req)
    else:
        planned_qty = net_req

    if planned_qty <= 0:
        continue

    machine = get_machine_by_series(series)
    if machine is None:
        continue

    time_hrs = get_time_required_by_series(series, planned_qty)
    if time_hrs is None:
        continue

    time_mins = time_hrs * 60
    used = machine_backlog.get(machine, 0)

    if used + time_mins <= MAX_MINUTES:
        machine_backlog[machine] = used + time_mins
        plan.append({
            "Machine": machine,
            "Series": series,
            "Part": part,
            "Qty": int(round(planned_qty)),
            "Time Required (hrs)": round(time_hrs, 2)
        })


In [ ]:
def get_cycle_time_by_series(series):
    s = norm(series)
    r = df_cycle[df_cycle["series"].astype(str).apply(norm) == s]

    if r.empty:
        return None

    cycle_time = r.iloc[0]["machine"]

    if pd.isna(cycle_time) or cycle_time <= 0:
        return None

    return cycle_time   # hours per unit


In [ ]:
def get_time_required_by_series(series, qty):
    cycle_time = get_cycle_time_by_series(series)
    if cycle_time is None:
        return None

    return qty * cycle_time   # hours


In [ ]:
time_hrs = get_time_required_by_series(series, planned_qty)
if time_hrs is None:
    continue


In [ ]:
rejection_log = []
PLAN = []
machine_load = {}
MAX_MINUTES = 22 * 60  # 22 hours

for i, row in df_master.iterrows():

    if i % 5000 == 0:
        print(f"Processed {i} rows...")

    series = row["series"]

    monthly_req = row["monthely_requirement-jan"] if not pd.isna(row["monthely_requirement-jan"]) else 0
    min_req     = row["minimum_requirement"] if not pd.isna(row["minimum_requirement"]) else 0
    dispatch    = row["dispatch"] if not pd.isna(row["dispatch"]) else 0

    fg_stock = get_fg_stock_by_series(series)
    available_fg = fg_stock - dispatch

    net_req = monthly_req + min_req - available_fg
    if net_req <= 0:
        rejection_log.append((series, "No net requirement"))
        continue

    daily_demand = monthly_req / 28 if monthly_req > 0 else 0

    if available_fg < daily_demand:
        planned_qty = daily_demand
    elif daily_demand <= 50:
        planned_qty = min(5 * daily_demand, net_req)
    else:
        planned_qty = net_req

    if planned_qty <= 0:
        rejection_log.append((series, "Planned qty <= 0"))
        continue

    machine = get_machine_by_series(series)
    if machine is None:
        rejection_log.append((series, "Machine not found"))
        continue

    cycle_time = get_cycle_time_by_series(series)
    if cycle_time is None:
        rejection_log.append((series, "Cycle time missing"))
        continue

    time_mins = planned_qty * cycle_time * 60
    used_mins = machine_load.get(machine, 0)

    if used_mins + time_mins > MAX_MINUTES:
        rejection_log.append((series, "Machine capacity exceeded"))
        continue

    machine_load[machine] = used_mins + time_mins

    PLAN.append({
        "Machine": machine,
        "Series": series,
        "Qty": int(round(planned_qty)),
        "Time Required (hrs)": round(time_mins / 60, 2)
    })


In [ ]:
df_reject = pd.DataFrame(rejection_log, columns=["Series", "Reason"])
df_reject["Count"] = 1

rejection_summary = (
    df_reject
    .groupby("Reason")
    .count()
    .sort_values("Count", ascending=False)
)

rejection_summary


In [ ]:
top_2_reasons = rejection_summary.head(2)
top_2_reasons
